In [0]:
%sql
CREATE TABLE IF NOT EXISTS datamodeling.silver.silver_table (
  order_id INT,
  order_date DATE,
  customer_id INT,
  customer_name STRING,
  customer_email STRING,
  product_id INT,
  product_name STRING,
  product_category STRING,
  quantity INT,
  unit_price DECIMAL(10,2),
  payment_type STRING,
  country STRING,
  last_updated DATE,
  customer_name_upper STRING,
  process_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW silver_source AS
WITH filtered AS (
    SELECT *
    FROM datamodeling.bronze.bronze_table
    WHERE last_updated > (
        SELECT COALESCE(MAX(last_updated), '1000-01-01')
        FROM datamodeling.silver.silver_table
    )
),
dedup AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY last_updated DESC
           ) AS rn
    FROM filtered
)
SELECT
    order_id,
    order_date,
    customer_id,
    customer_name,
    customer_email,
    product_id,
    product_name,
    product_category,
    quantity,
    unit_price,
    payment_type,
    country,
    last_updated,
    UPPER(customer_name) AS customer_name_upper,
    current_timestamp() AS process_ts
FROM dedup
WHERE rn = 1;

In [0]:
%sql
SELECT * FROM silver_source

In [0]:
%sql
MERGE INTO datamodeling.silver.silver_table t
USING silver_source s
ON t.order_id = s.order_id

WHEN MATCHED AND s.last_updated > t.last_updated THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *;

In [0]:
%sql
SELECT * FROM datamodeling.silver.silver_table